# Chapitre 2.5 — Prétraitement des mammographies

Une mammographie brute ne peut pas être envoyée telle quelle à GMIC : il attend des images de **taille fixe** (2944×1920), **sans le fond noir** autour du sein, dans une **plage d'intensité** précise. Cinq étapes transforment l'image brute en ce format.

**Tout le pipeline est écrit dans ce notebook** (seul le recadrage appelle le script original de GMIC). Il suffit d'indiquer un **dossier d'entrée** (les images à prétraiter) et un **dossier de sortie**, puis d'exécuter le notebook de haut en bas.

In [ ]:
import os, sys, glob, csv, pickle, shutil, zipfile, subprocess
from multiprocessing import Pool
import cv2, numpy as np, pydicom
import matplotlib.pyplot as plt
from course_utils import flowchart, gmic_dir, data_in, data_work

GMIC_DIR = gmic_dir()                           # sous-module modules/GMIC
if GMIC_DIR not in sys.path:
    sys.path.insert(0, GMIC_DIR)
GMIC_H, GMIC_W = 2944, 1920                     # taille d'entrée attendue par GMIC
VIEWS = ('L-CC', 'L-MLO', 'R-CC', 'R-MLO')

flowchart([
    '1. DICOM -> PNG 16-bit   (extrait les pixels, corrige MONOCHROME1)',
    '2. PKL GMIC              (liste examens + labels + chemins)',
    '3. Recadrage             (crop_mammogram.py de GMIC : retire le fond noir)',
    '4. Resize 2944x1920 + uint8',
    '5. Flip des vues droites (R-CC, R-MLO)',
], title='Pipeline de pretraitement GMIC')

## Quel dossier prétraiter ?

Renseigne `INPUT_DIR` (les images à traiter) et `OUTPUT_DIR` (où écrire le résultat), puis exécute le notebook. Le dossier d'entrée doit contenir `train_images/<patient>/<image>.dcm` et un `train.csv`.

- `data/in/rsna_sample` — les quelques images téléchargées au **chapitre 1**
- `data/in/rsna` — tout le dataset RSNA
- ou n'importe quel dossier au même format.

La sortie va dans `data/work/` (séparation entrée brute / résultats produits).

In [ ]:
# Dossier d'ENTRÉE : images à prétraiter (train_images/<pid>/<iid>.dcm + train.csv).
INPUT_DIR  = data_in('rsna_sample')   # <-- ex. data_in('rsna') pour tout
# Dossier de SORTIE : PNG prétraités + fichiers .pkl (dans data/work/).
OUTPUT_DIR = data_work('preprocess_sample')
CSV_PATH   = os.path.join(INPUT_DIR, 'train.csv')        # labels (patient / vue / cancer)
NUM_PROC   = 4                                           # cœurs pour la parallélisation

RAW_DIR     = os.path.join(INPUT_DIR, 'train_images')
PNG_DIR     = os.path.join(OUTPUT_DIR, 'png_images')
CROPPED_DIR = os.path.join(OUTPUT_DIR, 'cropped_images')
PKL_RAW     = os.path.join(OUTPUT_DIR, 'exam_list_before_cropping.pkl')
PKL_CROP    = os.path.join(OUTPUT_DIR, 'cropped_exam_list.pkl')
PKL_FINAL   = os.path.join(OUTPUT_DIR, 'data.pkl')
os.makedirs(PNG_DIR, exist_ok=True)

print('Entrée :', INPUT_DIR)
print('Sortie :', OUTPUT_DIR, '| processus:', NUM_PROC)
assert os.path.isdir(RAW_DIR), (
    f"{RAW_DIR} introuvable — indique un INPUT_DIR contenant train_images/, "
    f"ou exécute d'abord 01_download_data.ipynb.")

## Étape 1 — DICOM → PNG 16-bit

Les mammographies sont stockées en **DICOM** (`.dcm`), un format qui encapsule bien
plus que l'image (identité patient, paramètres scanner...). On en extrait les pixels
bruts vers des PNG 16-bit.

Piège classique : certains scanners stockent les niveaux **inversés**
(`PhotometricInterpretation == 'MONOCHROME1'`) — sans correction, le tissu mammaire
serait blanc sur fond noir, l'inverse de ce qu'attend GMIC.

In [ ]:
def dicom_to_png16(task):
    """Convertit 1 DICOM en PNG 16-bit (MONOCHROME1 inversé, normalisation [0, 65535])."""
    pid, iid = task
    out = os.path.join(PNG_DIR, pid, f'{iid}.png')
    if os.path.exists(out):
        return 'skip'
    zip_path = os.path.join(RAW_DIR, pid, f'{iid}.dcm.zip')
    dcm_path = os.path.join(RAW_DIR, pid, f'{iid}.dcm')
    try:
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path) as z:
                name = next(n for n in z.namelist() if n.endswith('.dcm'))
                with z.open(name) as fh:
                    ds = pydicom.dcmread(fh)
        elif os.path.exists(dcm_path):
            ds = pydicom.dcmread(dcm_path)
        else:
            return 'missing'
        arr = ds.pixel_array.astype(np.float32)
        if ds.PhotometricInterpretation == 'MONOCHROME1':
            arr = arr.max() - arr                      # inverse : blanc <-> noir
        m = arr.max()
        arr = (arr / m * 65535).astype(np.uint16) if m > 0 else arr.astype(np.uint16)
        os.makedirs(os.path.join(PNG_DIR, pid), exist_ok=True)
        cv2.imwrite(out, arr)
        return 'ok'
    except Exception as e:
        return f'err:{e}'

with open(CSV_PATH) as f:
    tasks = sorted({(r['patient_id'], r['image_id']) for r in csv.DictReader(f)})

with Pool(NUM_PROC) as pool:
    res = pool.map(dicom_to_png16, tasks)
print('Étape 1 — convertis:', res.count('ok'), '| déjà OK:', res.count('skip'),
      '| manquants/erreurs:', sum(1 for r in res if r not in ('ok', 'skip')))

## Étape 2 — Le fichier PKL GMIC

GMIC n'accède pas aux images directement : il lit un **`.pkl`** qui décrit chaque
examen — chemins des 4 vues (L-CC, L-MLO, R-CC, R-MLO), labels (`cancer_label`) et
métadonnées. On le construit depuis `train.csv`.

Convention RSNA des labels : `malignant` = sein avec cancer biopsié ; `benign` =
biopsié **sans** cancer. Un sein jamais biopsié reste NI l'un NI l'autre (ne pas
étiqueter ~98 % des seins sains comme « benign »).

In [ ]:
def build_exam_pkl(csv_path, png_dir, pkl_path):
    patients = {}
    with open(csv_path) as f:
        for r in csv.DictReader(f):
            pid = r['patient_id']
            patients.setdefault(pid, {
                'horizontal_flip': 'NO',
                'L-CC': [], 'L-MLO': [], 'R-CC': [], 'R-MLO': [],
                'cancer_label': {'benign': 0, 'left_benign': 0, 'right_benign': 0,
                                 'malignant': 0, 'left_malignant': 0,
                                 'right_malignant': 0, 'unknown': 0},
            })
            iid, lat, view = r['image_id'], r['laterality'], r['view']
            if not os.path.exists(os.path.join(png_dir, pid, f'{iid}.png')):
                continue
            vk = f'{lat}-{view}'
            if vk in patients[pid]:
                patients[pid][vk].append(f'{pid}/{iid}')
            side = 'left' if lat == 'L' else 'right'
            if int(r['cancer']) == 1:
                patients[pid]['cancer_label']['malignant'] = 1
                patients[pid]['cancer_label'][f'{side}_malignant'] = 1
            if int(r['biopsy']) == 1 and int(r['cancer']) == 0:
                patients[pid]['cancer_label']['benign'] = 1
                patients[pid]['cancer_label'][f'{side}_benign'] = 1
    exam_list = [e for e in patients.values() if any(e[v] for v in VIEWS)]
    with open(pkl_path, 'wb') as f:
        pickle.dump(exam_list, f)
    return exam_list

exams = build_exam_pkl(CSV_PATH, PNG_DIR, PKL_RAW)
print('Étape 2 — examens:', len(exams),
      '| dont cancer:', sum(1 for e in exams if e['cancer_label']['malignant']))

## Étape 3 — Recadrage (supprimer le fond noir)

Une mammographie brute contient beaucoup de **fond noir**. GMIC, entraîné sur des
images recadrées, est moins performant si on le garde.

On appelle ici le script **original de GMIC** `src/cropping/crop_mammogram.py` en
sous-processus (érosion → plus grande composante connexe → dilatation → bounding box).
C'est volontaire : on réutilise exactement la logique des auteurs du papier, plutôt
que de la réimplémenter. C'est aussi lui qui écrit la `window_location` dans le PKL.

In [ ]:
if os.path.exists(CROPPED_DIR):
    shutil.rmtree(CROPPED_DIR)

cmd = [sys.executable, 'src/cropping/crop_mammogram.py',
       '--input-data-folder', PNG_DIR,
       '--output-data-folder', CROPPED_DIR,
       '--exam-list-path', PKL_RAW,
       '--cropped-exam-list-path', PKL_CROP,
       '--num-processes', str(NUM_PROC)]
env = dict(os.environ, PYTHONPATH=GMIC_DIR + os.pathsep + os.environ.get('PYTHONPATH', ''))
res = subprocess.run(cmd, cwd=GMIC_DIR, env=env, stderr=subprocess.PIPE)
if res.returncode != 0:
    raise RuntimeError('crop_mammogram a échoué :\n' + res.stderr.decode()[-800:])
print('Étape 3 — images recadrées:',
      len(glob.glob(os.path.join(CROPPED_DIR, '**', '*.png'), recursive=True)))

## Étape 4 — Resize 2944×1920 + uint8

Après recadrage les images ont des tailles variables (selon le scanner). GMIC ayant
été entraîné en **2944×1920**, on redimensionne tout à cette taille puis on ramène en
**uint8** [0, 255].

**Deux interpolations** : `INTER_AREA` pour **réduire** (moyenne les pixels, évite
l'aliasing), `INTER_LINEAR` pour **agrandir** (bilinéaire, plus doux).

> Le pipeline de recherche met aussi à jour, dans le PKL, les coordonnées du crop
> (`rightmost_points`, etc.) à la nouvelle échelle — utile à l'inférence GMIC. On
> l'omet ici : ce notebook s'arrête aux images prêtes + `data.pkl`.

In [ ]:
def normalize_uint8(img):
    f = img.astype(np.float32); lo, hi = f.min(), f.max()
    return ((f - lo) / (hi - lo) * 255).astype(np.uint8) if hi > lo else np.zeros_like(f, np.uint8)

def resize_one(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return 'none'
    h, w = img.shape[:2]
    if (h, w) != (GMIC_H, GMIC_W):
        interp = cv2.INTER_AREA if (h > GMIC_H or w > GMIC_W) else cv2.INTER_LINEAR
        img = cv2.resize(img, (GMIC_W, GMIC_H), interpolation=interp)
    if img.dtype != np.uint8 or img.max() > 255:
        img = normalize_uint8(img)
    cv2.imwrite(path, img)
    return 'ok'

cropped_pngs = glob.glob(os.path.join(CROPPED_DIR, '**', '*.png'), recursive=True)
with Pool(NUM_PROC) as pool:
    res = pool.map(resize_one, cropped_pngs)
print('Étape 4 —', res.count('ok'), f'images en {GMIC_W}x{GMIC_H} uint8')

## Étape 5 — Flip horizontal des vues droites

GMIC a été entraîné avec **tous les seins orientés à gauche**. Les vues droites
(R-CC, R-MLO) sont naturellement orientées à droite : on les retourne une fois ici,
pour que toutes les images stockées soient déjà dans l'orientation attendue.

Puis `cropped_exam_list.pkl` est copié en **`data.pkl`** — le fichier final qu'on
passerait à l'inférence.

> La **normalisation mean/std** (z-score), elle, reste appliquée *à la volée* au
> chargement : elle produit des float négatifs, incompatibles avec un PNG uint8.

In [ ]:
def flip_one(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return 0
    cv2.imwrite(path, cv2.flip(img, 1))           # 1 = axe vertical = flip horizontal
    return 1

with open(PKL_CROP, 'rb') as f:
    exam_list = pickle.load(f)
right_paths = [os.path.join(CROPPED_DIR, sfp + '.png')
               for e in exam_list for v in ('R-CC', 'R-MLO')
               for sfp in e.get(v, [])
               if os.path.exists(os.path.join(CROPPED_DIR, sfp + '.png'))]
with Pool(NUM_PROC) as pool:
    flipped = sum(pool.map(flip_one, right_paths))
shutil.copy(PKL_CROP, PKL_FINAL)
print('Étape 5 — vues droites retournées:', flipped, '| data.pkl écrit ->', PKL_FINAL)

## Résultat

Chaque vue est désormais un PNG **2944×1920 uint8**, fond noir retiré, vues droites
alignées — directement exploitable par GMIC (chapitres 4 et 5). Vérifions sur une
vraie image.

In [ ]:
out_pngs = sorted(glob.glob(os.path.join(CROPPED_DIR, '**', '*.png'), recursive=True))
img = cv2.imread(out_pngs[0], cv2.IMREAD_UNCHANGED)
assert img.shape == (GMIC_H, GMIC_W) and img.dtype == np.uint8
print(f'✅ {len(out_pngs)} images prêtes pour GMIC — ex. {img.shape} {img.dtype}.')

rel = os.path.relpath(out_pngs[0], CROPPED_DIR)
src_png = os.path.join(PNG_DIR, rel)               # le PNG 16-bit avant recadrage
a, b = cv2.imread(src_png, cv2.IMREAD_UNCHANGED), img
fig, ax = plt.subplots(1, 2, figsize=(9, 6))
ax[0].imshow(a, cmap='gray'); ax[0].axis('off')
ax[0].set_title(f'PNG source\n{a.shape[1]}x{a.shape[0]}')
ax[1].imshow(b, cmap='gray'); ax[1].axis('off')
ax[1].set_title(f'Croppé + resize + flip\n{b.shape[1]}x{b.shape[0]} uint8')
plt.tight_layout(); plt.show()